In [12]:
import pandas as pd
import numpy as np

In [13]:
matches = pd.read_csv(
    "../data/final_matches_dataset.csv"
)

In [14]:
matches["Date"] = pd.to_datetime(
    matches["Date"]
)

In [15]:
matches = matches.sort_values(
    "Date"
)

In [16]:
teams = pd.concat([
    matches["home_team"],
    matches["away_team"]
]).unique()

In [17]:
new_features = [
    "home_last5_wins",
    "home_last5_draws",
    "home_last5_losses",
    "home_last5_goals",
    "home_last5_goals_conceded",
    "home_recent_form_score",

    "away_last5_wins",
    "away_last5_draws",
    "away_last5_losses",
    "away_last5_goals",
    "away_last5_goals_conceded",
    "away_recent_form_score"
]

for col in new_features:
    matches[col] = 0

In [18]:
def calculate_recent_form(
    matches_df,
    team,
    current_date,
    last_n=5
):

    previous_matches = matches_df[
        (
            (
                matches_df["home_team"] == team
            ) |
            (
                matches_df["away_team"] == team
            )
        ) &
        (
            matches_df["Date"] < current_date
        )
    ].sort_values(
        "Date",
        ascending=False
    ).head(last_n)

    wins = 0
    draws = 0
    losses = 0

    goals = 0
    conceded = 0

    for _, match in previous_matches.iterrows():

        # TEAM HOME
        if match["home_team"] == team:

            team_goals = match["home_score"]
            opponent_goals = match["away_score"]

        # TEAM AWAY
        else:

            team_goals = match["away_score"]
            opponent_goals = match["home_score"]

        goals += team_goals
        conceded += opponent_goals

        # RESULT
        if team_goals > opponent_goals:
            wins += 1

        elif team_goals < opponent_goals:
            losses += 1

        else:
            draws += 1

    form_score = (
        wins * 3 +
        draws * 1
    )

    return {
        "wins": wins,
        "draws": draws,
        "losses": losses,
        "goals": goals,
        "conceded": conceded,
        "form_score": form_score
    }

In [19]:
for idx, row in matches.iterrows():

    current_date = row["Date"]

    home_team = row["home_team"]
    away_team = row["away_team"]

    # HOME FORM
    home_form = calculate_recent_form(
        matches,
        home_team,
        current_date
    )

    # AWAY FORM
    away_form = calculate_recent_form(
        matches,
        away_team,
        current_date
    )

    # HOME
    matches.at[idx, "home_last5_wins"] = home_form["wins"]

    matches.at[idx, "home_last5_draws"] = home_form["draws"]

    matches.at[idx, "home_last5_losses"] = home_form["losses"]

    matches.at[idx, "home_last5_goals"] = home_form["goals"]

    matches.at[idx, "home_last5_goals_conceded"] = home_form["conceded"]

    matches.at[idx, "home_recent_form_score"] = home_form["form_score"]

    # AWAY
    matches.at[idx, "away_last5_wins"] = away_form["wins"]

    matches.at[idx, "away_last5_draws"] = away_form["draws"]

    matches.at[idx, "away_last5_losses"] = away_form["losses"]

    matches.at[idx, "away_last5_goals"] = away_form["goals"]

    matches.at[idx, "away_last5_goals_conceded"] = away_form["conceded"]

    matches.at[idx, "away_recent_form_score"] = away_form["form_score"]

In [20]:
matches["form_score_difference"] = (
    matches["home_recent_form_score"] -
    matches["away_recent_form_score"]
)

matches["recent_goals_difference"] = (
    matches["home_last5_goals"] -
    matches["away_last5_goals"]
)

matches["recent_defense_difference"] = (
    matches["away_last5_goals_conceded"] -
    matches["home_last5_goals_conceded"]
)

In [21]:
matches[[
    "home_team",
    "away_team",
    "home_recent_form_score",
    "away_recent_form_score",
    "form_score_difference"
]].head(20)

,home_team,away_team,home_recent_form_score,away_recent_form_score,form_score_difference
0,France,Mexico,0,0,0
1,United States,Belgium,0,0,0
2,Yugoslavia,Brazil,0,0,0
3,Romania,Peru,0,0,0
4,Argentina,France,0,3,-3
5,Chile,Mexico,0,0,0
6,Yugoslavia,Bolivia,3,0,3
7,United States,Paraguay,3,0,3
8,Uruguay,Peru,0,0,0
9,Argentina,Mexico,3,0,3


In [22]:
matches.to_csv(
    "../data/final_matches_with_form.csv",
    index=False
)

print("Dataset com forma recente salvo!")

Dataset com forma recente salvo!
